# CN3S Optimization

Prepare a saved rainfall/discharge snapshot, then load it separately for calibration.
Monthly inputs use month-start timestamps. Daily preparation is not implemented.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from cn3s import CN3S, CN3SOptimizer, CN3SParams
from cn3s.objectives import Objectives
from hydro.station import Station

## Configuration

Set `PREPARE_INPUTS = True` to create or replace the monthly snapshot using SQL
observations and the MERGE download cache. Leave it `False` to reuse saved inputs
without SQL or downloads. Use a continuous rainfall window, including the three antecedent months
used for warmup. Discharge means use the available daily values within the window.

In [2]:
DATA_ROOT = Path("/data/CN3S")
STATION_CODE = 29050000
STATION_DIR = DATA_ROOT / "stations" / str(STATION_CODE)
DOWNLOADS_DIR = DATA_ROOT / "downloads"
START_DATE = "2000-01-01"
END_DATE = "2026-09-30"
FREQ = "M"

PREPARE_INPUTS = True

## Load station context

In [3]:
station = Station.from_folder(STATION_DIR)
station.metadata

{'station_code': '29050000',
 'station_name': 'MARABÁ',
 'cobacia': '63911',
 'drainage_area': 703000.0,
 'drainage_area_geo': 696359.0281876093}

## Download and save discharge (optional)

`station.download_discharge(conn, ...)` saves `Monthly/discharge.parquet` using the
mean of available daily values, ignoring NaN. It does not retrieve rainfall.
The shared SQL connection remains caller-owned.

In [4]:
if PREPARE_INPUTS:
    from utils.sql_connector import SqlConnector

    if "conn" not in locals():
        conn = SqlConnector()

    station.download_discharge(
        conn,
        freq=FREQ,
        start=START_DATE,
        end=END_DATE,
    )

To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code BAK85UR77 to authenticate.


## Download and save rainfall (optional)

`station.download_rain(downloader, ...)` clips MERGE to the basin and saves monthly
rainfall totals in `Monthly/rain.parquet`. It does not use a SQL connection.
Both frequency folders are created; daily downloading remains unimplemented.

In [5]:
if PREPARE_INPUTS:
    from mergedownloader.downloader import Downloader
    from mergedownloader.file_downloader import DownloadMode, FileDownloader
    from mergedownloader.inpeparser import InpeParsers

    downloader = Downloader(
        file_downloader=FileDownloader(download_mode=DownloadMode.NO_UPDATE),
        local_folder=str(DOWNLOADS_DIR),
        parsers=InpeParsers,
    )
    station.download_rain(
        downloader,
        freq=FREQ,
        start=START_DATE,
        end=END_DATE,
    )

Using HTTP downloads


## Load and inspect saved optimization inputs

`get_optim_data(FREQ)` loads both saved series and joins discharge to the rainfall
calendar. Missing discharge remains NaN; no downloads or SQL queries occur.

In [7]:
inputs = station.get_optim_data(FREQ)
inputs.tail()

,prec_mm,q_m3s
date,,
2026-05-01,6.500626,10529.482816
2026-06-01,4.328246,5197.531222
2026-07-01,4.803958,3315.959915
2026-08-01,0.736519,2022.291857
2026-09-01,33.027168,1364.446962


## Configure optimization

Use the inventory drainage area (`drainage_area`, km²), matching the previous workflow.
The saved metadata also contains `drainage_area_geo` for explicit comparison.

ACT behavior remains unchanged: positive ACT shifts simulated timestamps by days,
and the optimizer matches observations by exact date. With monthly inputs this can
leave positive-ACT candidates without matches; addressing that limitation is deferred.

In [8]:
params = CN3SParams(
    name=station.name,
    area=station.metadata["drainage_area"],
    r0=60,
    cn_i=19,
    alfa=0.2,
    beta=0.01,
    k0=.38,
    k1=.36,
    k2=.68,
    act=1,
    warmup_steps=3,
)

model = CN3S(params, freq=FREQ)

In [9]:
model.run(inputs["prec_mm"])
model.evaluate(inputs["q_m3s"])

Running CN3S model:   0%|          | 0/317 [00:00<?, ?step/s]

Aligned 297 time steps between model results and observations.
From 2000-05-01 00:00:00 to 2026-09-01 00:00:00.


-2.7952070804804516

In [10]:
model.plot()

In [ ]:
params.bounds["act"] = (0, 2)

In [16]:
optim = CN3SOptimizer(
    prec=inputs["prec_mm"],
    observed_q=inputs["q_m3s"],
    model=model,
    train_ratio=0.99,
    optimize_params=["r0", "cn_i", "beta", "alfa", "k0", "k1", "k2", "act"],
    objective=Objectives.ExtremeFlowBlend,
)

print(f"Aligned record: {len(optim.aligned)} time steps")
print(f"Train: {optim.train_idx} | Test: {len(optim.aligned) - optim.train_idx}")

Aligned record: 301 time steps
Train: 297 | Test: 4


## Run calibration

In [17]:
# Interrupt at any time — optim.model will hold the best parameters seen so far.
# x0 = [63.98, 20.71, 0.33, 0.01, 0.38, 0.36, 0.68, 1]
x0 = None
optim.optimize(maxiter=1000, tol=1e-4, disp=False, workers=-1, x0=x0)

/usr/local/lib/python3.12/dist-packages/scipy/optimize/_differentialevolution.py:487: UserWarning:

differential_evolution: the 'workers' keyword has overridden updating='immediate' to updating='deferred'



1 - Train objective: 0.9022 - {'r0': 700.6686145471804, 'cn_i': 30.827478260501877, 'beta': 0.00619933952014351, 'alfa': 0.884178830119123, 'k0': 0.7940266425865702, 'k1': 0.03513321590494001, 'k2': 0.3330406056020365, 'act': 0.0}
2 - Train objective: 0.9022 - {'r0': 700.6686145471804, 'cn_i': 30.827478260501877, 'beta': 0.00619933952014351, 'alfa': 0.884178830119123, 'k0': 0.7940266425865702, 'k1': 0.03513321590494001, 'k2': 0.3330406056020365, 'act': 0.0}
3 - Train objective: 0.9022 - {'r0': 700.6686145471804, 'cn_i': 30.827478260501877, 'beta': 0.00619933952014351, 'alfa': 0.884178830119123, 'k0': 0.7940266425865702, 'k1': 0.03513321590494001, 'k2': 0.3330406056020365, 'act': 0.0}
4 - Train objective: 0.9022 - {'r0': 700.6686145471804, 'cn_i': 30.827478260501877, 'beta': 0.00619933952014351, 'alfa': 0.884178830119123, 'k0': 0.7940266425865702, 'k1': 0.03513321590494001, 'k2': 0.3330406056020365, 'act': 0.0}
5 - Train objective: 0.5011 - {'r0': 75.04601764269671, 'cn_i': 6.2537190693

In [19]:
model.run(inputs["prec_mm"])
model.evaluate(inputs["q_m3s"])
model.plot()

Running CN3S model:   0%|          | 0/317 [00:00<?, ?step/s]

Aligned 297 time steps between model results and observations.
From 2000-05-01 00:00:00 to 2026-09-01 00:00:00.


In [18]:
optim.save(station.station_dir/"optim/ExtremeFlowBlend_alfa_livre.json")

In [15]:
model.params

CN3SParams(name='MARABÁ', area=703000.0, r0=113.53575898046515, cn_i=1.0, alfa=0.011310494820731912, beta=0.004157016423653198, k0=1.0, k1=0.2918575560424726, k2=0.32939715148671983, act=1, warmup_steps=3)